In [1]:
from agents.KnowledgeExtractAgent import KEA
from agents.FactorConstructAgent import FCA

d:\Software\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_PATH = 'data/sample1.pdf'
QUERY = "Constrcut a factor"
MODEL = 'DeepSeek-V3.2'

In [3]:
kea = KEA()
instruction = kea.extract_knowledge(PDF_PATH, QUERY, MODEL)

In [4]:
instruction#[2]

[{'factor_name': 'BASE_factor',
  'expression': 'add(mul(0.7, rank(div(1, Clsprc))), mul(0.3, neg(zscore(ts_sum(ChangeRatio, 10)))))',
  'core_logic': 'Convex combination of value (inverse price rank) and reversal (negated 10-day return z-score).',
  'data_source': ['Clsprc', 'ChangeRatio']},
 {'factor_name': 'REGIME_filter',
  'expression': 'gt(ts_mean(gt(ChangeRatio, 0), 63), 0.60)',
  'core_logic': 'Binary regime indicator for when the fraction of positive return days over the trailing 63 days exceeds 60%.',
  'data_source': ['ChangeRatio']},
 {'factor_name': 'EDGE_signal',
  'expression': 'mul(add(mul(0.7, rank(div(1, Clsprc))), mul(0.3, neg(zscore(ts_sum(ChangeRatio, 10))))), gt(ts_mean(gt(ChangeRatio, 0), 63), 0.60))',
  'core_logic': 'BASE factor multiplied by the REGIME filter to activate the signal only during drift conditions.',
  'data_source': ['Clsprc', 'ChangeRatio']}]

In [4]:
fca = FCA()
db = fca.handle_instruction(instruction[0])

In [5]:
db

{'ok': True,
 'df_factors': [                   BASE_factor
  Trddt      Stkcd              
  2020-01-15 23         0.457809
             39         0.613561
             89         0.516762
             156        0.448004
             409        0.170829
  ...                        ...
  2024-12-31 603918     0.567579
             603950    -0.078052
             605008     0.207665
             605136     0.230984
             605499    -0.084808
  
  [112153 rows x 1 columns]]}

In [9]:
fca.factor_names

['BASE_factor']

In [10]:
fca.backtest('BASE_factor', db['df_factor_panel']).mean()

np.float64(-0.00023353758702155235)